# Step 1 — Audit the raw dataset

#### Imports

In [14]:
import pandas as pd
import numpy as np

#### Load

In [15]:
path = '../data/raw/scrapped_data.csv'
real_estate = pd.read_csv(path)
real_estate.head()

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Ownership,Car_Parking,Bathroom,Balcony,Super_Area,Dimensions,Under_Construction
0,3 BHK Flat for Sale in Saarrthi Shimmer n Shi...,1.75 Cr,"11,305 per sqft",1165 sqft,Ready to Move,6 out of 12,Resale,Furnished,East,Main Road,Saarrthi Shimmer n Shine,Co-operative Society,2 Covered,3,2.0,NaN,NaN,NaN
1,"3 BHK Flat for Sale in Acolade, Kharadi, Pune",1.52 Cr,"8,444 per sqft",1400 sqft,Ready to Move,7 out of 12,Resale,Semi-Furnished,East,Garden/Park,Acolade,Freehold,2 Covered,3,2.0,NaN,NaN,NaN
2,"4 BHK Flat for Sale in Windsor Avenue, Wanowr...",2.60 Cr,"11,064 per sqft",NaN,Ready to Move,4 out of 9,Resale,Semi-Furnished,East,Main Road,Windsor Avenue,Co-operative Society,2 Covered,5,3.0,2350 sqft,NaN,NaN
3,"1 BHK Flat for Sale in Raheja Sterling, NIBM ...",61.6 Lac,"8,107 per sqft",560 sqft,Ready to Move,1 out of 18,Resale,Unfurnished,East,"Garden/Park, Main Road",Raheja Sterling,Co-operative Society,1 Covered,1,NaN,NaN,NaN,NaN
4,"3 BHK House for Sale in Kalpana Empress, Wano...",2.75 Cr,"13,750 per sqft",NaN,Ready to Move,NaN,Resale,Furnished,East,Main Road,Kalpana Empress,Co-operative Society,3 Covered,4,2.0,2000 sqft,NaN,NaN


#### Get the overall structure

In [16]:
real_estate.info()

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               930 non-null    str    
 1   Price               930 non-null    str    
 2   Rate_per_SqFt       820 non-null    str    
 3   Carpet_Area         846 non-null    str    
 4   Status              735 non-null    str    
 5   Floor               770 non-null    str    
 6   Transaction         930 non-null    str    
 7   Furnishing          929 non-null    str    
 8   facing              776 non-null    str    
 9   overlooking         756 non-null    str    
 10  Society             811 non-null    str    
 11  Ownership           776 non-null    str    
 12  Car_Parking         554 non-null    str    
 13  Bathroom            930 non-null    int64  
 14  Balcony             867 non-null    float64
 15  Super_Area          84 non-null     str    
 16  Dimensions         

In [17]:
real_estate.isna().sum()

Title                   0
Price                   0
Rate_per_SqFt         110
Carpet_Area            84
Status                195
Floor                 160
Transaction             0
Furnishing              1
facing                154
overlooking           174
Society               119
Ownership             154
Car_Parking           376
Bathroom                0
Balcony                63
Super_Area            846
Dimensions            906
Under_Construction    749
dtype: int64

In [18]:
real_estate.duplicated().sum()

np.int64(3)

Duplicate records

In [19]:
real_estate[real_estate.duplicated(keep=False)]

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Ownership,Car_Parking,Bathroom,Balcony,Super_Area,Dimensions,Under_Construction
144,"4 BHK Apartment for Sale in Godrej Ivara, Khar...",3.09 Cr,NaN,1550 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Godrej Ivara,NaN,NaN,4,4.0,NaN,NaN,Poss. by Dec '31
162,"4 BHK Apartment for Sale in Godrej Ivara, Khar...",3.09 Cr,NaN,1550 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Godrej Ivara,NaN,NaN,4,4.0,NaN,NaN,Poss. by Dec '31
184,3 BHK Apartment for Sale in Mantra Melange Kha...,1.55 Cr,NaN,1010 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Mantra Melange Kharadi Riverside,NaN,NaN,3,2.0,NaN,NaN,Poss. by Dec '29
185,3 BHK Apartment for Sale in Mantra Melange Kha...,1.55 Cr,NaN,1010 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Mantra Melange Kharadi Riverside,NaN,NaN,3,2.0,NaN,NaN,Poss. by Dec '29
186,3 BHK Apartment for Sale in Mantra Melange Kha...,1.55 Cr,NaN,1010 sqft,NaN,NaN,New Property,Unfurnished,NaN,NaN,Mantra Melange Kharadi Riverside,NaN,NaN,3,2.0,NaN,NaN,Poss. by Dec '29


These are ducplicate rows (duplicate listings)
so retaining just one copy is enough. 

In [20]:
real_estate.duplicated().sum(), real_estate.shape

(np.int64(3), (930, 18))

In [21]:
real_estate = real_estate.drop_duplicates(keep='first')

In [22]:
real_estate.duplicated().sum(), real_estate.shape

(np.int64(0), (927, 18))

Missing value percentage per column

In [23]:
pd.concat([real_estate.isna().sum().rename('NaN_cnt'), round(real_estate.isna().mean() * 100, 2).rename('PerMissing_pct')], axis = 1).sort_values(by = 'PerMissing_pct', ascending= False)

,NaN_cnt,PerMissing_pct
Dimensions,903,97.41
Super_Area,843,90.94
Under_Construction,749,80.80
Car_Parking,373,40.24
Status,192,20.71
overlooking,171,18.45
Floor,157,16.94
facing,151,16.29
Ownership,151,16.29
Society,119,12.84


### hidden missing values

In [24]:
missing_tokens = ["", "nan", "none", "n/a", "null", "-"]

text_df = real_estate.select_dtypes(include="str")
sus_counts = text_df.apply(
    lambda col: col.str.strip().str.lower().isin(missing_tokens).sum()
)
sus_counts


Title                 0
Price                 0
Rate_per_SqFt         0
Carpet_Area           0
Status                0
Floor                 0
Transaction           0
Furnishing            0
facing                0
overlooking           0
Society               0
Ownership             0
Car_Parking           0
Super_Area            0
Dimensions            0
Under_Construction    0
dtype: int64

#### Selecting Columns to Drop.

1. Dimensions
97.42% missing

In [25]:
real_estate.Dimensions.value_counts(dropna= False).head()

Dimensions
NaN                  903
40 X 50 ft Sqft        2
45 X 28 ft Sqft        1
80 X 50 ft Sqft        1
100 X 115 ft Sqft      1
Name: count, dtype: int64

Insights
- seems mostly consistent in format (length * width)
- not in directly usable numeric format
- after removing duplicates NaN value count is 903

Do the values provide information that isn't already captured by Carpet_Area or Super_Area?

In [26]:
real_estate.loc[
    real_estate.Dimensions.notna(),
    ['Dimensions', 'Carpet_Area', 'Super_Area']
].head(10)

,Dimensions,Carpet_Area,Super_Area
6,45 X 28 ft Sqft,2400 sqft,NaN
120,80 X 50 ft Sqft,3000 sqft,NaN
128,100 X 115 ft Sqft,2000 sqft,NaN
241,40 X 50 ft Sqft,2825 sqft,NaN
307,78 X 22.44 ft Sqft,5440 sqft,NaN
335,1000 X 1.5 ft Sqft,1600 sqft,NaN
340,45 X 55 ft Sqft,3000 sqft,NaN
341,50 X 42.98 ft Sqft,2965 sqft,NaN
370,50 X 60 ft Sqft,5000 sqft,NaN
378,60 X 58.33 ft Sqft,1800 sqft,NaN


Insight
- Super_Area is missing for 22 of them
- Dimensions don't appear consistent with other Area values. For example, 45 × 28 = 1,260 square feet, while the corresponding Carpet_Area is 2,400 sqft


##### Decision
Drop Dimensions from this cleaning project<br>
Justification :
- It contains approx 97% missing values
- Remaining values contains inconsistent text instead of numeric measurement
- Interpretation relative to other area columns is uncertain

In [27]:
real_estate = real_estate.drop(columns= 'Dimensions')
real_estate.head()

,Title,Price,Rate_per_SqFt,Carpet_Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Ownership,Car_Parking,Bathroom,Balcony,Super_Area,Under_Construction
0,3 BHK Flat for Sale in Saarrthi Shimmer n Shi...,1.75 Cr,"11,305 per sqft",1165 sqft,Ready to Move,6 out of 12,Resale,Furnished,East,Main Road,Saarrthi Shimmer n Shine,Co-operative Society,2 Covered,3,2.0,NaN,NaN
1,"3 BHK Flat for Sale in Acolade, Kharadi, Pune",1.52 Cr,"8,444 per sqft",1400 sqft,Ready to Move,7 out of 12,Resale,Semi-Furnished,East,Garden/Park,Acolade,Freehold,2 Covered,3,2.0,NaN,NaN
2,"4 BHK Flat for Sale in Windsor Avenue, Wanowr...",2.60 Cr,"11,064 per sqft",NaN,Ready to Move,4 out of 9,Resale,Semi-Furnished,East,Main Road,Windsor Avenue,Co-operative Society,2 Covered,5,3.0,2350 sqft,NaN
3,"1 BHK Flat for Sale in Raheja Sterling, NIBM ...",61.6 Lac,"8,107 per sqft",560 sqft,Ready to Move,1 out of 18,Resale,Unfurnished,East,"Garden/Park, Main Road",Raheja Sterling,Co-operative Society,1 Covered,1,NaN,NaN,NaN
4,"3 BHK House for Sale in Kalpana Empress, Wano...",2.75 Cr,"13,750 per sqft",NaN,Ready to Move,NaN,Resale,Furnished,East,Main Road,Kalpana Empress,Co-operative Society,3 Covered,4,2.0,2000 sqft,NaN


In [28]:
real_estate.shape

(927, 17)

### Investigate Super_Area

In [29]:
real_estate.Super_Area.value_counts(dropna=False).head()

Super_Area
NaN          843
1500 sqft      4
1600 sqft      2
1775 sqft      2
1650 sqft      2
Name: count, dtype: int64

- Has 843 missing values after removing duplicates

In [30]:
real_estate.loc[
    real_estate.Super_Area.notna(),
    ['Super_Area', 'Carpet_Area']
# ]
].head(10)

,Super_Area,Carpet_Area
2,2350 sqft,NaN
4,2000 sqft,NaN
9,2845 sqft,NaN
11,3200 sqft,NaN
18,1500 sqft,NaN
33,780 sqft,NaN
38,1045 sqft,NaN
89,1092 sqft,NaN
99,3000 sqft,NaN
105,1600 sqft,NaN


Observation / Pattern
- Every row containing `Super_Area` has a missing `Carpet_Area`.
- Two columns may be alternative ways of reporting a property's area in this dataset.

#### Make the decision

In [31]:
real_estate.loc[
    real_estate.Super_Area.notna() & real_estate.Carpet_Area.notna(),
    ['Super_Area', 'Carpet_Area']
]

,Super_Area,Carpet_Area


In [32]:
real_estate.loc[
    real_estate.Super_Area.isna() & real_estate.Carpet_Area.isna(),
    ['Super_Area', 'Carpet_Area']
]

,Super_Area,Carpet_Area


That gives us a clean partition:
- Every row has Super_Area populated, or
- Every row has Carpet_Area populated,
- But never both and never neither.

Decision: <br>
For now lets keep it as it is .
<br>
Later, after typecasting to numeric values, we can consider creating a unified analytical column with a clearly documented fallback rule.

### Investigate `Under_Construction` column

In [33]:
real_estate.Under_Construction.value_counts(dropna= False).head()

Under_Construction
NaN                 749
Poss. by Dec '27     25
Poss. by Dec '26     22
Poss. by Dec '28     22
Poss. by Dec '29     13
Name: count, dtype: int64

Observation
- 749 missing values.
- The remaining records contain possession estimates such as Poss. by Dec '27, Poss. by Dec '26, and Poss. by Dec '30.
- These values appear to represent expected possession dates, not simply a Yes/No construction status. <br> <br>


This column may tell us when a property is expected to be ready for possession, which could be useful when analyzing prices and availability

#### Decision
Keep it for now<br>

Reason:
- Contains expected possession dates
- Could help distinguish listings with future possession dates
- Losing potentially useful timing information

### final audit before Step 2

In [34]:
missing_tokens = ["", "nan", "none", "n/a", "null", "-"]

real_estate.select_dtypes(include="str").apply(
    lambda col: col.str.strip().str.lower().isin(missing_tokens).sum()
).sum()

np.int64(0)

---